# Feature Engineering – Interaktives Notebook

## 1. Imports & Daten laden

In [ ]:
import pandas as pd
import sqlite3
import numpy as np
import sys
import os

# Workspace-Root zum Pfad hinzufügen damit database.py gefunden wird
sys.path.insert(0, os.path.abspath(".."))
from database import DB_PATH

In [ ]:
def get_df_field():
    conn = sqlite3.connect(DB_PATH)
    query_field_players = """
    SELECT ps.*, p.position
    FROM player_stats_field ps
    JOIN players p ON ps.player_id = p.id
"""
    df_field_players = pd.read_sql_query(query_field_players, conn)
    conn.close()
    df_field_players = df_field_players.sort_values(by=['player_id', 'season', 'matchday']).reset_index(drop=True)
    df_field_players["minutes"] = pd.to_numeric(df_field_players["minutes"], errors="coerce")
    return df_field_players

def get_df_gk():
    conn = sqlite3.connect(DB_PATH)
    query_goalkeeper = "SELECT * FROM player_stats_gk"
    df_goalkeeper = pd.read_sql_query(query_goalkeeper, conn)
    conn.close()
    df_goalkeeper = df_goalkeeper.sort_values(by=['player_id', 'season', 'matchday']).reset_index(drop=True)
    df_goalkeeper["minutes"] = pd.to_numeric(df_goalkeeper["minutes"], errors="coerce")
    return df_goalkeeper

In [ ]:
df_field = get_df_field()
df_gk = get_df_gk()
print(f"Feldspieler: {df_field.shape}")
print(f"Torhueter:   {df_gk.shape}")
df_field.head(5)

## 2. Feature-Funktionen

In [ ]:
def points_avg_and_trend(df):
    df["points_filled"] = df['points'].fillna(0)
    df["points_avg_last_3"] = (
        df.groupby(["player_id", "season"])["points_filled"]
        .shift(1).rolling(window=3).mean()
    )
    df["points_avg_last_5"] = (
        df.groupby(["player_id", "season"])["points_filled"]
        .shift(1).rolling(window=5).mean()
    )
    df["points_trend"] = df["points_avg_last_3"] - df["points_avg_last_5"]
    df = df.drop(columns=["points_filled"])
    return df

In [ ]:
df_field = points_avg_and_trend(df_field)
df_field[["player_id", "matchday", "season", "points", "points_avg_last_3", "points_avg_last_5", "points_trend"]].head(20)

In [ ]:
def minutes_avg_and_trend(df):
    df["minutes_filled"] = df["minutes"].fillna(0)
    df["minutes_avg_last_3"] = (
        df.groupby(["player_id", "season"])["minutes_filled"]
        .shift(1).rolling(window=3).mean()
    )
    df["minutes_avg_last_5"] = (
        df.groupby(["player_id", "season"])["minutes_filled"]
        .shift(1).rolling(window=5).mean()
    )
    df["has_played"] = np.where(df["minutes_avg_last_3"].fillna(0) > 0, 1, 0)
    df = df.drop(columns=["minutes_filled"])
    return df

In [ ]:
df_field = minutes_avg_and_trend(df_field)
df_field[["player_id", "matchday", "minutes", "minutes_avg_last_3", "minutes_avg_last_5", "has_played"]].head(20)

In [ ]:
def points_p90(df):
    df["points_p90_last_5"] = (df["points_avg_last_5"] / df["minutes_avg_last_5"]) * 90
    df["points_p90_last_5"] = df["points_p90_last_5"].replace([np.inf, -np.inf], np.nan).fillna(0)
    df["points_p90_last_3"] = (df["points_avg_last_3"] / df["minutes_avg_last_3"]) * 90
    df["points_p90_last_3"] = df["points_p90_last_3"].replace([np.inf, -np.inf], np.nan).fillna(0)
    return df

In [ ]:
df_field = points_p90(df_field)
df_field[["player_id", "matchday", "points_avg_last_3", "points_avg_last_5", "points_p90_last_3", "points_p90_last_5"]].head(20)

In [ ]:
def p90(df, efficiency_cols):
    for col in efficiency_cols:
        df[f"{col}_filled"] = df[col].fillna(0)
        df[f"{col}_avg_last_3"] = (
            df.groupby(["player_id", "season"])[f"{col}_filled"]
            .shift(1).rolling(window=3).mean()
        )
        df[f"{col}_p90_last_3"] = df[f"{col}_avg_last_3"] / df["minutes_avg_last_3"] * 90
        df[f"{col}_p90_last_3"] = df[f"{col}_p90_last_3"].replace([np.inf, -np.inf], np.nan).fillna(0)
        df = df.drop(columns=[f"{col}_filled", f"{col}_avg_last_3"])
    return df

In [ ]:
def ratios(df, ratio_pairs):
    for succes_col, total_col in ratio_pairs:
        df[f"{succes_col}_filled"] = df[succes_col].fillna(0)
        df[f"{total_col}_filled"] = df[total_col].fillna(0)
        df[f"{succes_col}_sum_last_3"] = (
            df.groupby(["player_id", "season"])[f"{succes_col}_filled"]
            .shift(1).rolling(window=3).sum()
        )
        df[f"{total_col}_sum_last_3"] = (
            df.groupby(["player_id", "season"])[f"{total_col}_filled"]
            .shift(1).rolling(window=3).sum()
        )
        df[f"{succes_col}_ratio_last_3"] = df[f"{succes_col}_sum_last_3"] / df[f"{total_col}_sum_last_3"]
        df[f"{succes_col}_ratio_last_3"] = df[f"{succes_col}_ratio_last_3"].replace([np.inf, -np.inf], np.nan).fillna(0)
        df = df.drop(columns=[
            f"{succes_col}_filled", f"{succes_col}_sum_last_3",
            f"{total_col}_filled", f"{total_col}_sum_last_3"
        ])
    return df

In [ ]:
def form_trends(df, form_trend_cols):
    for col in form_trend_cols:
        df[f"{col}_filled"] = df[col].fillna(0)
        df[f"{col}_avg_last_3"] = (
            df.groupby(["player_id", "season"])[f"{col}_filled"]
            .shift(1).rolling(window=3).mean()
        )
        df[f"{col}_avg_last_5"] = (
            df.groupby(["player_id", "season"])[f"{col}_filled"]
            .shift(1).rolling(window=5).mean()
        )
        df[f"{col}_trend"] = df[f"{col}_avg_last_3"] - df[f"{col}_avg_last_5"]
        df = df.drop(columns=[f"{col}_filled"])
    return df

In [ ]:
def sums(df, sum_cols):
    df["cards_total"] = df["yellow_cards"].fillna(0) + df["yellow_red_cards"].fillna(0) * 2 + df["red_cards"].fillna(0) * 3
    current_sum_cols = sum_cols + ["cards_total"]
    for col in current_sum_cols:
        df[f"{col}_sum_last_5"] = (
            df.groupby(["player_id", "season"])[col]
            .shift(1).rolling(window=5).sum().fillna(0)
        )
    df = df.drop(columns=["cards_total"])
    return df

In [ ]:
def lag_features(df, lag_cols, lag=1):
    for col in lag_cols:
        if col not in df.columns:
            continue
        lagged = df.groupby(["player_id", "season"])[col].shift(lag)
        df[f"{col}_lag_{lag}"] = pd.to_numeric(lagged).fillna(0)
    return df

In [ ]:
def map_status(df):
    status_categories = [
        "startelf", "bank", "fit_nicht_gespielt", "nicht_beruecksichtigt",
        "angeschlagen", "aufbautraining", "verletzt",
        "gelb_gesperrt", "gelbrot_gesperrt", "rot_gesperrt", "unbekannt",
    ]
    df["status"] = df["status"].fillna("unbekannt")
    df.loc[~df["status"].isin(status_categories), "status"] = "unbekannt"
    df["status"] = pd.Categorical(df["status"], categories=status_categories)
    status_dummies = pd.get_dummies(df["status"], prefix="status")
    df = pd.concat([df.drop(columns=["status"]), status_dummies], axis=1)
    return df

In [ ]:
def create_target_variable(df):
    df["target_points"] = df.groupby(["player_id", "season"])["points"].shift(-1)
    return df

## 3. Konfigurations-Listen

In [ ]:
# --- Feldspieler ---
efficiency_cols_field = [
    "gewonnene_zweikaempfe", "schuesse_aufs_tor", "torschuss_vorlagen",
    "kreierte_grosschancen", "geklaerte_baelle", "abgefangene_baelle",
    "balleroberungen", "geblockte_baelle", "fehler_vor_gegentor",
    "begangene_fouls", "ballverluste"
]

ratio_pairs_field = [
    ("erfolgreiche_paesse", "paesse_gesamt"),
    ("gewonnene_zweikaempfe", "gewonnene_zweikaempfe_gesamt"),
    ("gewonnene_luftkaempfe", "gewonnene_luftkaempfe_gesamt"),
    ("erfolgreiche_tacklings", "tacklings_gesamt"),
    ("erfolgreiche_dribblings", "dribblings_gesamt"),
    ("schussgenauigkeit", "schussgenauigkeit_gesamt")
]

form_trend_cols_field = [
    "goals_own_team", "goals_enemy_team", "match_result",
    "grade", "points_per_value", "ligaInsider_points"
]

sum_cols_field = ["goals", "assists"]

lag_cols_field = [
    "points", "red_cards", "yellow_red_cards", "yellow_cards",
    "goals", "assists", "erfolgreiche_paesse", "gewonnene_zweikaempfe", "grade"
]

print(f"efficiency_cols_field: {len(efficiency_cols_field)} Spalten")
print(f"ratio_pairs_field:     {len(ratio_pairs_field)} Paare")
print(f"form_trend_cols_field: {len(form_trend_cols_field)} Spalten")
print(f"lag_cols_field:        {len(lag_cols_field)} Spalten")

In [ ]:
# --- Torhueter ---
efficiency_cols_gk = ["paraden", "fehler_vor_gegentor"]

ratio_pairs_gk = [
    ("abgewehrte_schuesse", "abgewehrte_schuesse_gesamt"),
    ("strafraum_beherrschung", "strafraum_beherrschung_gesamt"),
    ("abgewehrte_elfmeter", "elfmeter_gesamt"),
    ("grosschancen_pariert", "grosschancen_gesamt")
]

form_trend_cols_gk = [
    "goals_own_team", "goals_enemy_team", "match_result", "grade", "points_per_value"
]

sum_cols_gk = ["weisse_weste"]

lag_cols_gk = ["points", "red_cards", "yellow_red_cards", "yellow_cards", "grade"]

print(f"efficiency_cols_gk: {len(efficiency_cols_gk)} Spalten")
print(f"ratio_pairs_gk:     {len(ratio_pairs_gk)} Paare")
print(f"form_trend_cols_gk: {len(form_trend_cols_gk)} Spalten")
print(f"lag_cols_gk:        {len(lag_cols_gk)} Spalten")

## 4. Pipeline

In [ ]:
def process_data(df):
    df = points_avg_and_trend(df)
    df = minutes_avg_and_trend(df)
    df = points_p90(df)
    df = create_target_variable(df)
    df = map_status(df)

    is_field_player = df["position"].iloc[0] != 1

    if is_field_player:
        df = form_trends(df, form_trend_cols_field)
        df = p90(df, efficiency_cols_field)
        df = ratios(df, ratio_pairs_field)
        df = sums(df, sum_cols_field)
        df = lag_features(df, lag_cols_field)
    else:
        df = form_trends(df, form_trend_cols_gk)
        df = p90(df, efficiency_cols_gk)
        df = ratios(df, ratio_pairs_gk)
        df = sums(df, sum_cols_gk)
        df = lag_features(df, lag_cols_gk)

    return df

In [ ]:
def get_final_ml_data(df, is_training=True):
    if df is None or df.empty:
        print("Empty DataFrame an get_final_ml_data übergeben")
        return df

    if "market_value" in df.columns:
        df["market_value"] = pd.to_numeric(df["market_value"], errors="coerce")
    if "points_per_value" in df.columns:
        df["points_per_value"] = pd.to_numeric(df["points_per_value"], errors="coerce")
    if "position" not in df.columns:
        df["position"] = 1

    df = process_data(df)

    if is_training:
        df = df.dropna(subset=["target_points"])
    else:
        df = df.groupby("player_id").tail(1).reset_index(drop=True)
        df = df.drop(columns=["target_points"], errors="ignore")

    conn = sqlite3.connect(DB_PATH)
    df_team_stats = pd.read_sql_query("SELECT * FROM team_stats", conn)
    conn.close()
    df_team_stats = df_team_stats.sort_values(by=["team_name", "season", "matchday"])

    df = pd.merge(df, df_team_stats, how="left", on=["season", "matchday", "team_name"], suffixes=("", "_team"))
    df = pd.merge(df, df_team_stats, how="left",
                  left_on=["season", "matchday", "opponent_name"],
                  right_on=["season", "matchday", "team_name"], suffixes=("", "_opponent"))

    drop_cols_field = [
        "id", "player_id", "team_name", "opponent_name", "date", "points", "minutes",
        "goals_own_team", "goals_enemy_team", "match_result",
        "goals", "assists", "yellow_cards", "yellow_red_cards", "red_cards",
        "ligaInsider_points", "grade",
        "erfolgreiche_paesse", "paesse_gesamt",
        "gewonnene_zweikaempfe", "gewonnene_zweikaempfe_gesamt",
        "gewonnene_luftkaempfe", "gewonnene_luftkaempfe_gesamt",
        "erfolgreiche_tacklings", "tacklings_gesamt",
        "begangene_fouls", "geklaerte_baelle", "abgefangene_baelle",
        "balleroberungen", "ballverluste",
        "erfolgreiche_dribblings", "dribblings_gesamt",
        "torschuss_vorlagen", "kreierte_grosschancen",
        "schuesse_aufs_tor", "schussgenauigkeit", "schussgenauigkeit_gesamt",
        "fehler_vor_gegentor", "geblockte_baelle",
        "team_name_team", "team_name_opponent", "id_team", "id_opponent",
        "points_per_minute"
    ]

    drop_cols_opponent = [col for col in df.columns if "opponent_2" in col or "opponent_3" in col]
    drop_cols_opponent.append("opponent_1_opponent")
    drop_cols_opponent.append("opponent_1")

    drop_cols_gk = [
        "id_opponent", "id_team", "id", "player_id", "team_name", "opponent_name",
        "date", "points", "minutes",
        "goals_own_team", "goals_enemy_team", "match_result",
        "goals", "assists", "yellow_cards", "yellow_red_cards", "red_cards",
        "ligaInsider_points", "grade",
        "team_name_team", "team_name_opponent",
        "abgewehrte_schuesse", "abgewehrte_schuesse_gesamt",
        "paraden", "weisse_weste",
        "strafraum_beherrschung", "strafraum_beherrschung_gesamt",
        "abgewehrte_elfmeter", "elfmeter_gesamt",
        "grosschancen_pariert", "grosschancen_gesamt",
        "fehler_vor_gegentor", "points_per_minute"
    ]

    if df["position"].iloc[0] != 1:
        df = df.drop(columns=drop_cols_field, errors="ignore")
        df = df.drop(columns=drop_cols_opponent, errors="ignore")
    else:
        df = df.drop(columns=drop_cols_gk, errors="ignore")
        df = df.drop(columns=drop_cols_opponent, errors="ignore")

    return df

In [ ]:
def split_by_position(df):
    df_def = df[df["position"] == 2].copy()
    df_mid = df[df["position"] == 3].copy()
    df_off = df[df["position"] == 4].copy()
    df_def = df_def.drop(columns=["position"])
    df_mid = df_mid.drop(columns=["position"])
    df_off = df_off.drop(columns=["position"])
    return df_def, df_mid, df_off

## 5. Pipeline ausführen & Features inspizieren

In [ ]:
df_field_raw = get_df_field()
df_field_ml = get_final_ml_data(df_field_raw, is_training=True)
print(f"Shape nach Pipeline: {df_field_ml.shape}")
df_field_ml.head(5)

In [ ]:
# Alle Spalten die ans Modell gehen
print(f"Anzahl Spalten gesamt: {len(df_field_ml.columns)}")
feature_cols = [c for c in df_field_ml.columns if c != "target_points"]
print(f"Feature-Spalten (ohne target): {len(feature_cols)}")
print("\nSpalten:")
for col in sorted(df_field_ml.columns):
    marker = "  [TARGET]" if col == "target_points" else ""
    print(f"  {col}{marker}")

In [ ]:
# Statistiken der Features
df_field_ml[feature_cols].describe()

In [ ]:
# GK Pipeline
df_gk_raw = get_df_gk()
df_gk_raw["position"] = 1
df_gk_ml = get_final_ml_data(df_gk_raw, is_training=True)
print(f"Shape nach GK-Pipeline: {df_gk_ml.shape}")
print("\nGK-Features:")
for col in sorted(df_gk_ml.columns):
    marker = "  [TARGET]" if col == "target_points" else ""
    print(f"  {col}{marker}")